# merge_variants: merged models of the literature comparison (CPU, no GPU needed)

Builds the merges of the two single-teacher students at lr 1e-4 and 5400 steps (BGE-M3: 66.07, 8K-embed: 65.67) that the comparison with the joint two-teacher run needs (`arastirma/literatur_yontemleri_plani.md`, A1 and A2): the linear soup at three weights, SLERP of the two changes from the base, and TIES merging with 50 % and 20 % of every change kept. Reference: the joint run `scale_two_lr1e4_mix200k_s5400` (TR-MTEB 67.16); at lr 5e-5 the soup scored 66.52 against 66.71 for the joint run. Each merge takes a few minutes on a CPU runtime and is written to `models/<name>`; finished merges are skipped. Evaluate at least `lit_a1_soup_bge50`, `lit_a2_slerp_delta50` and `lit_a2_ties_d50` first. Delete this notebook when the results are in.

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

import glob, os
from huggingface_hub import snapshot_download
from turkish_distill.soup import merge_models

def latest_final(name):
    dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final'))
    assert dirs, f'no finished run of {name}'
    return dirs[-1]

BGE = latest_final('scale_bge_mix200k_s5400_lr1e4')       # BGE-M3 single teacher, lr 1e-4, 5400 steps (TR-MTEB 66.07)
EIGHT_K = latest_final('scale_8k_mix200k_s5400_lr1e4')    # 8K-embed single teacher, same recipe (TR-MTEB 65.67)
# the model both students started from (the changes from it are what TIES and the delta SLERP work on), pinned revision
BASE_DIR = snapshot_download('ytu-ce-cosmos/modernbert-tr-base-1k', revision='c2e35f553101cce5b03fbb318ef8728473fd34dd', local_dir='/content/base1k')

# (name, method, weights of (BGE, 8K), extra arguments)
MERGES = [
    ('lit_a1_soup_bge50', 'linear', [1, 1], {}),
    ('lit_a1_soup_bge30', 'linear', [3, 7], {}),
    ('lit_a1_soup_bge70', 'linear', [7, 3], {}),
    ('lit_a2_slerp_delta50', 'slerp', [1, 1], dict(base=BASE_DIR)),
    ('lit_a2_ties_d50', 'ties', [1, 1], dict(base=BASE_DIR, density=0.5)),
    ('lit_a2_ties_d20', 'ties', [1, 1], dict(base=BASE_DIR, density=0.2)),
]
for name, method, weights, kwargs in MERGES:
    out = f'{DRIVE_ROOT}/models/{name}'
    if os.path.exists(f'{out}/SOUP.json'):
        print(f'{name}: exists, skipped')
        continue
    print(f'{name}: {method} {weights}')
    merge_models([BGE, EIGHT_K], out, method, weights, label=name, **kwargs)

print('\nEval presets (paste into an eval notebook):')
for name, *_ in MERGES:
    print(f"    '{name}': dict(model=f'{{DRIVE_ROOT}}/models/{name}'),")

In [ ]:
from google.colab import runtime
runtime.unassign()